In [ ]:
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from neo4j import GraphDatabase

print("="*60)
print(" INITIATING AUTONOMOUS GRAPH EXTRACTION PIPELINE ")
print("="*60)

# 1. Secure Authentication
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "AlphaFund2026!"

# 2. The Unstructured Data Stream (Mocking Week 4 Transcripts)
raw_document_chunks = [
    "Tony's Q3 Compliance Report detailed the massive Alpha AI buyout by Microsoft.",
    "Satya Nadella, its CEO, approved the $2B deal yesterday morning.",
    "Meanwhile, Apple announced they have heavily invested in a new startup called NeuralNet Labs.",
    "Tim Cook mentioned that NeuralNet Labs directly competes with Alpha AI in the generative space."
]
print(f"[System] Loaded {len(raw_document_chunks)} unstructured text chunks.")

In [ ]:
# Step 2: The LLM Compiler Setup
# (Assuming the Pydantic classes KnowledgeGraph, Node, and Edge from Part 3 are already defined in the cell above)

print("[System] Initializing Gemini 2.5 Flash with Constrained Decoding...")

# 1. Lock the LLM to Temperature 0.0 for deterministic extraction
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

# 2. Bind the Pydantic Schema
graph_extractor = llm.with_structured_output(KnowledgeGraph)

# 3. The Compiler Prompt (Using the exact logic from Part 4)
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an elite Information Extraction AI. Extract entities and relationships. "
               "Resolve all pronouns to canonical names. Use ONLY allowed relationship types. "
               "Output MUST strictly adhere to the JSON schema."),
    ("human", "Extract the graph from this text:\n\n{text}")
])

# 4. The LangChain Pipeline
extraction_chain = extraction_prompt | graph_extractor

[System] Initializing Gemini 2.5 Flash with Constrained Decoding...


NameError: name 'ChatGoogleGenerativeAI' is not defined

In [ ]:
# Step 3: The Cypher Translation Engine
def push_graph_to_neo4j(graph_data: KnowledgeGraph, driver):
    """Translates a Pydantic Graph object into idempotent Cypher transactions."""

    with driver.session() as session:
        # 1. Process all Nodes First
        for node in graph_data.nodes:
            # We dynamically inject the Label, but parameterize the properties
            node_query = f"""
            MERGE (n:{node.label} {{id: $id}})
            ON CREATE SET n.original_text = $original_text
            """
            session.run(node_query, id=node.id, original_text=node.original_text)

        # 2. Process all Edges
        for edge in graph_data.edges:
            # We match the source and target nodes, then MERGE the edge between them
            edge_query = f"""
            MATCH (source {{id: $source_id}})
            MATCH (target {{id: $target_id}})
            MERGE (source)-[r:{edge.relationship_type}]->(target)
            """
            session.run(edge_query,
                        source_id=edge.source_node_id,
                        target_id=edge.target_node_id)

In [ ]:
# Step 4: The Main Execution Loop
print("\n[System] Commencing Extraction & Ingestion Loop...")

# Open the persistent connection to Docker
neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

try:
    for i, chunk in enumerate(raw_document_chunks):
        print(f"\n--- Processing Chunk {i+1}/{len(raw_document_chunks)} ---")
        print(f"Text: '{chunk}'")

        # Phase 1: AI Extraction (Unstructured -> Structured)
        extracted_graph = extraction_chain.invoke({"text": chunk})

        node_count = len(extracted_graph.nodes)
        edge_count = len(extracted_graph.edges)
        print(f"Extraction Success: Found {node_count} Nodes and {edge_count} Edges.")

        # Phase 2: Database Ingestion (Structured -> Graph DB)
        push_graph_to_neo4j(extracted_graph, neo4j_driver)
        print(f"Ingestion Success: Committed to Neo4j.")

except Exception as e:
    print(f"\n[FATAL ERROR] Pipeline halted: {str(e)}")

finally:
    neo4j_driver.close()
    print("\n[System] Pipeline execution complete. Driver connection closed.")

In [ ]:
# Step 5: The Verification Query
# --- THE ULTIMATE VERIFICATION TEST ---
print("\n" + "="*40)
print(" VERIFYING GRAPH INTEGRITY ")
print("="*40)

def verify_graph():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

    # Query: Find companies that directly compete with companies acquired by Microsoft
    test_query = """
    MATCH (c1:Company)-[:COMPETES_WITH]->(c2:Company)<-[:ACQUIRED]-(:Company {id: 'Microsoft'})
    RETURN c1.id AS Competitor, c2.id AS Acquired_Startup
    """

    with driver.session() as session:
        results = session.run(test_query)
        for record in results:
            print(f"Graph Intelligence: {record['Competitor']} competes directly with {record['Acquired_Startup']}, which is owned by Microsoft.")

    driver.close()

verify_graph()